In [32]:
import pandas as pd

student_info = pd.read_csv("../data/raw/studentInfo.csv")
student_vle = pd.read_csv("../data/raw/studentVle.csv")

In [33]:
import os
import matplotlib.pyplot as plt

os.makedirs("../figures", exist_ok=True)

def save_figure(filename):
    plt.tight_layout()
    plt.savefig(
        f"../figures/{filename}",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

In [34]:
engagement = (
    student_vle
    .groupby("id_student")["sum_click"]
    .sum()
    .reset_index()
)

In [35]:
model_df = student_info.merge(
    engagement,
    on="id_student",
    how="left"
)

model_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,sum_click
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2158.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,1034.0


In [36]:
model_df.shape
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  str    
 1   code_presentation     32593 non-null  str    
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  str    
 4   region                32593 non-null  str    
 5   highest_education     32593 non-null  str    
 6   imd_band              31482 non-null  str    
 7   age_band              32593 non-null  str    
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  str    
 11  final_result          32593 non-null  str    
 12  sum_click             29741 non-null  float64
dtypes: float64(1), int64(3), str(9)
memory usage: 3.2 MB


In [37]:
model_df["success"] = (
    model_df["final_result"]
    .isin(["Pass", "Distinction"])
    .astype(int)
)

model_df["success"].value_counts()

success
0    17208
1    15385
Name: count, dtype: int64

In [38]:
# Handle missing values
# imd_band has missing values, so we keep them as their own category

model_df["imd_band"] = model_df["imd_band"].fillna("Unknown")

In [39]:
model_df.columns

Index(['code_module', 'code_presentation', 'id_student', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
       'studied_credits', 'disability', 'final_result', 'sum_click',
       'success'],
      dtype='str')

In [40]:
# Rename sum_click to total_clicks if needed
if "sum_click" in model_df.columns and "total_clicks" not in model_df.columns:
    model_df = model_df.rename(columns={"sum_click": "total_clicks"})

# Fill missing values
model_df["total_clicks"] = model_df["total_clicks"].fillna(0)
model_df["imd_band"] = model_df["imd_band"].fillna("Unknown")

# Create target if needed
model_df["success"] = (
    model_df["final_result"]
    .isin(["Pass", "Distinction"])
    .astype(int)
)

features = [
    "gender",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
    "total_clicks"
]

# Check for missing columns before selecting
missing_cols = [col for col in features if col not in model_df.columns]
print("Missing columns:", missing_cols)

X = model_df[features]
y = model_df["success"]

X_encoded = pd.get_dummies(X, drop_first=True)

print("Feature matrix shape:", X_encoded.shape)
print("Target shape:", y.shape)

X_encoded.head()

Missing columns: []
Feature matrix shape: (32593, 21)
Target shape: (32593,)


,num_of_prev_attempts,studied_credits,total_clicks,gender_M,highest_education_HE Qualification,highest_education_Lower Than A Level,highest_education_No Formal quals,highest_education_Post Graduate Qualification,imd_band_10-20,imd_band_20-30%,...,imd_band_40-50%,imd_band_50-60%,imd_band_60-70%,imd_band_70-80%,imd_band_80-90%,imd_band_90-100%,imd_band_Unknown,age_band_35-55,age_band_55<=,disability_Y
0,0,240,934.0,True,True,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
1,0,60,1435.0,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
2,0,60,281.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,True
3,0,60,2158.0,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
4,0,60,1034.0,False,False,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False


In [41]:
import os

os.listdir("../Data/Processed")


['student_success_modeling_data.csv']

In [42]:
processed_df = X_encoded.copy()
processed_df["success"] = y.values

processed_df.to_csv(
    "../Data/processed/student_success_modeling_data.csv",
    index=False
)

print("Dataset saved.")

Dataset saved.


In [43]:
processed_df.to_csv(
    "../Data/processed/student_success_modeling_data.csv",
    index=False
)

In [44]:
X_encoded.columns = (
    X_encoded.columns
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "lt", regex=False)
    .str.replace(">", "gt", regex=False)
    .str.replace("%", "pct", regex=False)
    .str.replace("-", "_", regex=False)
)